# 7 · JSON for machines, YAML for people

**The problem.** An office directory is served to a web app as JSON and maintained by the operations team as a YAML
file in git. Two classic bugs are waiting:

- **The Norway problem.** In YAML 1.1, which many YAML libraries still implement or offer, the unquoted country code
  `NO` reads as the boolean `false`. So do `no`, `off` and `n`, and `yes`/`on`/`y` read as `true`.
- **Leading zeros.** Oslo's postal code `0150` reads as the number 150, or as octal. `1:30` reads as ninety
  (sexagesimal).

JSON has its own traps: JavaScript's `JSON.parse` turns every number into a float, so `2**60` quietly loses
precision, and `JSON.stringify` writes `NaN` and `Infinity` as `null`.

Since case study 6 all data passes through plain data with exact types, so the text layer only has to write and read
that faithfully. `JSON` and `YAML` mirror `Plain` exactly: `ToJSON(store)(schema, value)`,
`FromJSON(store)(schema, text)`, with `.Reachable` for graphs. The framework's `JSON` module is imported here as `Json`, so it doesn't
shadow JavaScript's global `JSON`.

In [1]:
import * as Y from "yaml"; // the YAML package, only to show what a stock YAML 1.1 reader does

import { Errors, JSON as Json, Proxies, Schemas, Validators, YAML } from "@mbse/schemas/Framework";

const store = new Proxies.OfStore();

const { DecodeError } = Errors;

let Office = new Schemas.OfObject.Builder().ref()
  .properties(
    (p) => p.name("city").of((t) => t.as_native(String)),
    (p) => p.name("country_code").of((t) => t.as_native(String)), // ISO 3166-1 alpha-2
    (p) => p.name("postal_code").of((t) => t.as_native(String)),
    (p) => p.name("opens").of((t) => t.as_native(String)), // local time, 'HH:MM'
    (p) => p.name("desks").of((t) => t.as_native(BigInt)),
    (p) => p.name("occupancy").of((t) => t.as_native(Number)),
    (p) => p.name("open_plan").of((t) => t.as_native(Boolean)),
  )
  .create();
let Staff = new Schemas.OfObject.Builder().ref().properties((p) => p.name("name").of((t) => t.as_native(String))).create();
const Seating = new Schemas.OfRelation.Builder()
  .links("office", "person")
  .properties((p) => p.name("desk").of((t) => t.as_native(String)))
  .unique("office") // each person sits in one office
  .create();
Office = new Schemas.OfObject.Builder(Office).relations((adj) => adj.name("staff").of(Seating).me("office")).update();
Staff = new Schemas.OfObject.Builder(Staff).relations((adj) => adj.name("office").of(Seating).me("person")).update();
store.register("Office", Office);
store.register("Staff", Staff);
store.register("Seating", Seating);

const oslo = store.Office()
  .city("Oslo").country_code("NO").postal_code("0150").opens("1:30")
  .desks(2n ** 60n).occupancy(NaN).open_plan(false)
  .staff((x: any) => x.person((p: any) => p.name("Ingrid")).desk("A-01"))
  .create();

## JSON

`Json.ToJSON` writes a snapshot as JSON text. `indent` is optional:

In [2]:
const text = Json.ToJSON(store).Reachable(Office, oslo, { indent: 2 });
console.log(text);

{
  "root": "s0",
  "objects": {
    "s0": {
      "city": "Oslo",
      "country_code": "NO",
      "postal_code": "0150",
      "opens": "1:30",
      "desks": 1152921504606846976,
      "occupancy": "NaN",
      "open_plan": false,
      "staff": [
        {
          "person": {
            "$ref": "s1",
            "$schema": "Staff"
          },
          "desk": "A-01"
        }
      ]
    },
    "s1": {
      "name": "Ingrid",
      "office": [
        {
          "office": {
            "$ref": "s0",
            "$schema": "Office"
          },
          "desk": "A-01"
        }
      ]
    }
  }
}


- `"NO"`, `"0150"` and `"1:30"` are strings because the schema says so. JSON quotes strings anyway.
- `NaN` became the *string* `"NaN"`, so the output is strict JSON that any parser accepts. The float schema knows to
  read it back as NaN.
- `2n ** 60n` is written exactly, and read back as a `bigint`: the framework's JSON reader keeps integers and floats
  apart, which `JSON.parse` can't.
- Key order follows the schema and the snapshot, and non-ASCII text is written as is.

Reading is strict in the same spirit. Text that other parsers accept with a shrug is refused here, because accepting
it would mean guessing:

In [3]:
const load = Json.FromJSON(store);
for (const [why, bad] of [
  ["NaN literal", '{"root": "s0", "objects": {"s0": {"occupancy": NaN}}}'],
  ["duplicate key", '{"root": "s0", "root": "s1", "objects": {}}'],
  ["int for a float", '{"root": "s0", "objects": {"s0": {"occupancy": 1}}}'],
  ["truncated", '{"root": "s0", "objects": '],
] as const) {
  try {
    load(Office, bad);
  } catch (error) {
    if (!(error instanceof DecodeError)) throw error;
    console.log(`${why.padEnd(16)} -> ${error.message}`);
  }
}

NaN literal      -> line 1, column 48: NaN is not valid JSON


duplicate key    -> line 1, column 16: duplicate key 'root'


int for a float  -> $.objects.s0.occupancy: expected float, got int


truncated        -> line 1, column 27: expecting a value


Every problem is a `DecodeError` (case study 6), located where it was found. Problems in the *text* (a `NaN`
literal, a duplicate key, a truncated document) have a line and column. The wrong type is only found once the text
has become plain data and meets the schema, so it has a path. Columns count characters (code points), not bytes or
UTF-16 units, so an editor's "go to line and column" lands on the spot.

**Why so strict about `1` for a float?** Because another language may have separate integer and float types. If `1`
were quietly accepted as `1.0`, the same document could mean different things in TypeScript and in Python. Duplicate
keys are refused for the same reason: parsers disagree about which one wins.

The output is also **byte-identical** across implementations. The Python binding writes exactly the same text for
the same data, which is checked by a shared conformance corpus. You can diff, hash or sign a document without
caring which language produced it.

In [4]:
const back = load.Reachable(Office, text) as Proxies.Instance;
console.log(back.country_code, back.postal_code, back.opens, back.desks === 2n ** 60n, Number.isNaN(back.occupancy));

NO 0150 1:30 true true


## YAML

`YAML.ToYAML` writes the same snapshot as YAML:

In [5]:
console.log(YAML.ToYAML(store).Reachable(Office, oslo));

root: s0
objects:
  s0:
    city: Oslo
    country_code: 'NO'
    postal_code: '0150'
    opens: '1:30'
    desks: 1152921504606846976
    occupancy: NaN
    open_plan: false
    staff:
    - person:
        $ref: s1
        $schema: Staff
      desk: A-01
  s1:
    name: Ingrid
    office:
    - office:
        $ref: s0
        $schema: Office
      desk: A-01



Every string a YAML reader might misread is quoted: `'NO'`, `'0150'`, `'1:30'`, `'NaN'`. The rule is "quote if a
YAML 1.1 **or** 1.2 reader could take it for anything but this string", so the file is safe whichever library the
next reader uses. A stock YAML 1.1 reader (the `yaml` package in 1.1 mode) reads it back correctly:

In [6]:
const stock = Y.parse(YAML.ToYAML(store).Reachable(Office, oslo), { version: "1.1" });
console.log(stock.objects.s0.country_code, stock.objects.s0.postal_code, stock.objects.s0.opens);

NO 0150 1:30


## Hand-written YAML

People don't quote carefully. Here is the ops team's file, typed by hand:

In [7]:
const handwritten = `
root: s0
objects:
  s0:
    city: Oslo
    country_code: NO        # YAML 1.1: false. YAML 1.2: the string "NO"
    postal_code: 0150       # YAML 1.1: 104 (octal). YAML 1.2 core: 150, an int
    opens: 1:30             # YAML 1.1: 90. YAML 1.2: the string "1:30"
    desks: 40
    open_plan: false
`;
console.log("stock YAML 1.1 reader:", Y.parse(handwritten, { version: "1.1" }).objects.s0);

stock YAML 1.1 reader: {
  city: "Oslo",
  country_code: false,
  postal_code: 104,
  opens: 90,
  desks: 40,
  open_plan: false
}


The framework loads YAML with the **YAML 1.2 core schema**. Only `true`/`false` are booleans, and `1:30` and dates
are strings. That fixes `NO` and `1:30`. `0150` is still an integer in YAML 1.2, and the schema says `postal_code`
is a string. Since types are never coerced, loading refuses rather than turning 150 into `"150"` and silently losing
the zero:

In [8]:
const fromYAML = YAML.FromYAML(store);
try {
  fromYAML(Office, handwritten);
} catch (error) {
  console.log(String(error));
}

DecodeError: $.objects.s0.postal_code: expected str, got int


Quote it, and it loads:

In [9]:
const office = fromYAML(Office, handwritten.replace("0150 ", "'0150'")) as Proxies.Instance;
console.log(office.country_code, office.postal_code, office.opens, office.open_plan);
console.log(Validators.Validate(store)(Office, office));

NO 0150 1:30 false


[]


That's the pattern for configuration files: load with the framework, and a wrong type is an error with a location,
never a quiet reinterpretation. YAML features that don't map to plain data are refused too: tags like `!!binary`,
several documents in one file, and non-string keys. Anchors and aliases are accepted on input and never written.

In [10]:
for (const [why, bad] of [
  ["tag", "root: s0\nobjects:\n  s0:\n    city: !!binary AAAA\n"],
  ["two documents", "---\nroot: s0\n---\nroot: s1\n"],
  ["non-string key", "1: x\n"],
] as const) {
  try {
    fromYAML(Office, bad);
  } catch (error) {
    console.log(`${why.padEnd(15)} -> ${(error as Error).message}`);
  }
}

tag             -> line 4, column 11: unsupported tag !!binary


two documents   -> line 3, column 1: expected a single document, found another


non-string key  -> line 1, column 1: keys must be strings, got int 1


These are the framework's own YAML rules, so their messages are the same in every language binding. Plain syntax
errors (an unclosed bracket, a tab used for indentation) are reported as a `DecodeError` with a line and column too,
but the wording comes from each binding's YAML parser:

In [11]:
try {
  fromYAML(Office, "root: [s0\n");
} catch (error) {
  console.log(String(error));
}

DecodeError: line 2, column 1: flow sequence in block collection must be sufficiently indented and end with a ]


## Plain data in, plain data out

Both modules also expose `loads`/`dumps` for plain data without a schema. They're handy for inspecting files, or for
converting between the two formats:

In [12]:
const plain = YAML.loads(YAML.ToYAML(store).Reachable(Office, oslo));
console.log(Json.dumps(plain) === Json.ToJSON(store).Reachable(Office, oslo));

true


## What you learned

- `JSON` and `YAML` mirror `Plain`: `ToJSON`/`ToYAML(store)(schema, value)`, `FromJSON`/`FromYAML(store)(schema, text)`,
  and `.Reachable`.
- JSON output is strict (non-finite floats become strings), exact for big integers, and byte-identical across
  language bindings.
- YAML is written so both 1.1 and 1.2 readers agree, and read with the 1.2 core schema. The Norway problem is gone.
- Types are never coerced on input. A value of the wrong type is an error, not a guess.
- Every input problem is a `DecodeError`: a line and column for text, or a path for plain data.

**Next:** so far you've used tools the framework provides. Case study 8 writes new ones, a data dictionary generated
from schemas and an audit log that works for any object, and shows why the visitor protocol makes that possible.